# NLP embeddings and topic labels (Role C · Sentence-BERT)

**Idea.** B's clusters use TF-IDF, which only sees exact words: *"I gained a lot of weight"* and *"I put on pounds"* share no important word. **Sentence-BERT** turns each review into a vector of **768 numbers that captures its meaning**, so similar reviews get similar vectors. We use `FremyCompany/BioLORD-2023`, trained on biomedical text. This is the **new technique that was not taught in the course** (the course covers LSH, clustering and frequent items, but not text embeddings).

**What it is used for.**
1. Compare clustering by meaning (k-means on the embeddings) with B's 100 TF-IDF clusters: Davies-Bouldin index, **drug purity**, **ARI** and rating spread, also after hiding the drug names.
2. Describe each of B's clusters with keywords: c-TF-IDF (written by hand), then a KeyBERT-style selection with the embeddings. Evaluated with automatic metrics; there is no manual labelling.
3. Give A one vector per `review_id` for the similar-story search by meaning (cosine LSH).

This serves research question 1: *can patient reviews be grouped into common experience types?* Outputs, usage instructions and limitations are in the README.

## Pipeline overview

| Step | Section | What it does |
|---|---|---|
| Load & scope | C1 | Load A's clean data and B's cluster assignments, keep the four conditions; reload the embeddings computed in an earlier Kaggle session |
| Embeddings | C2 | Sentence-BERT vectors per condition (optionally also for all other reviews) |
| Sanity check | C3 | Rows, IDs, NaN and vector length agree |
| Topic terms | C4 | Class-based TF-IDF from scratch: the most distinctive terms of every B cluster |
| Topic keywords | C5 | KeyBERT-style: the c-TF-IDF candidates ranked by similarity to the cluster centroid, with diversity (MMR) |
| Cluster table | C6 | Reviews, mean rating and top drugs of each of B's 100 clusters |
| Automatic validation | C7 | NPMI coherence vs random baseline; embedding cohesion |
| DBSCAN | C8 | Why only the k-means clusters are described |
| Clustering in meaning space | C9 | k-means on embeddings vs B's TF-IDF clusters (DB, drug purity, ARI), with and without drug names |
| Save outputs | C10 | Embeddings for A, labels for D, statistics for the report |

## 0 · Setup and parameters

All settings live here. Change a value, re-run from top, and every downstream cell picks it up automatically.

In [1]:
import json
import os
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)
NOTEBOOK_STARTED = time.perf_counter()

# where the data is read from and written to  (same layout as Roles A, B and E)
INPUT_DIRS = [Path('/kaggle/input'), Path('data'), Path('../data'), Path('../clustering/outputs')]
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_stats = {}   # every reported number lands here; saved as JSON at the end

# ── Scope ─────────────────────────────────────────────────────────────────────
CONDITIONS = ['depression', 'birth control', 'pain', 'acne']   # B's four conditions (lower case)
TEXT, ID, COND = 'review_text_clean', 'review_id', 'condition_name'
CLUSTER_COL = 'kmeans_cluster'         # DBSCAN is not used for the description (see C8)

# ── Sentence-BERT ─────────────────────────────────────────────────────────────
MODEL_NAME = 'FremyCompany/BioLORD-2023'   # biomedical Sentence-BERT (768-d). To go back: 'all-MiniLM-L6-v2' (384-d)
MODEL_SLUG = 'biolord'                      # goes into every file name, so two models never share a cache
EMB        = f'emb_{MODEL_SLUG}'            # embeddings per condition
EMB_MASKED = f'emb_{MODEL_SLUG}_masked_v2'  # same, drug names masked
EMB_OTHER  = f'emb_{MODEL_SLUG}_other'      # other conditions
BATCH_SIZE = 64
EMBED_ALL_OTHER_REVIEWS = False  # also embed the ~92k reviews of other conditions (A's story search uses all 128k). Slow with BioLORD (hours on CPU); True if you have a GPU or time

# ── Labelling ─────────────────────────────────────────────────────────────────
N_CENTRAL, N_RANDOM = 5, 5       # central and random reviews shown per cluster

key = lambda c: c.replace(' ', '_')

## C1 · Load and scope

I load A's clean dataset (`reviews_clean_deduplicated.parquet`) and B's cluster assignments (`cluster_assignments.parquet`), and create `cond_l` (the condition in lower case, without extra spaces) so that filtering by condition does not fail because of capitalisation.
I only work with 4 conditions (Depression, Birth Control, Pain, Acne), because those are the ones B grouped into clusters: 36,365 reviews and 100 k-means clusters (Birth Control 23, Depression 26, Pain 26, Acne 25).


In [2]:
def find_input(file_name):
    for folder in INPUT_DIRS:
        if folder.exists():
            matches = sorted(folder.rglob(file_name))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'{file_name} not found in {INPUT_DIRS}')

reviews  = pd.read_parquet(find_input('reviews_clean_deduplicated.parquet'))   # Role A
clusters = pd.read_parquet(find_input('cluster_assignments.parquet'))          # Role B
reviews['cond_l'] = reviews[COND].str.lower().str.strip()
run_stats['n_reviews_all'] = int(len(reviews))
run_stats['n_reviews_in_scope'] = int(reviews['cond_l'].isin(CONDITIONS).sum())

print(reviews.shape)
print(reviews.columns.tolist())
print()
print("Reviews per condition (my 4):")
print(reviews[reviews["cond_l"].isin(CONDITIONS)]["cond_l"].value_counts())
print()
print("B's k-means clusters per condition:")
print(clusters.groupby(COND)[CLUSTER_COL].nunique())

(128663, 11)
['review_id', 'drug_name', 'drug_name_aliases', 'condition_name', 'rating', 'review_date', 'useful_count', 'review_text_clean', 'review_text_normalised', 'word_count', 'cond_l']

Reviews per condition (my 4):
cond_l
birth control    19520
depression        7109
pain              5187
acne              4549
Name: count, dtype: int64

B's k-means clusters per condition:
condition_name
Acne             25
Birth Control    23
Depression       26
Pain             26
Name: kmeans_cluster, dtype: int64


### C1b · Reload the precomputed embeddings

Encoding the reviews with BioLORD takes a long time on a CPU, so the embeddings were computed once in an earlier Kaggle session with the code of C2 (`FremyCompany/BioLORD-2023`, `normalize_embeddings=True`, `max_seq_length` 128) and stored in a public Kaggle dataset: https://www.kaggle.com/datasets/claudiagranado/dataset-input-versionantiguaresultados

The cell below loads `embeddings_all.npy`, `embeddings_masked_v2_all.npy` and their `_ids.parquet` files, splits them per condition in the row order of A's dataset, and saves the per-condition files that C2–C9 expect. Because those files then exist, C2 skips the encoding. To recompute them from scratch, run the notebook without this dataset attached and delete this cell; C2 and C9 will then encode the reviews (hours on a CPU).


In [3]:
# Reload the embeddings computed in an earlier Kaggle session (they are stored in a Kaggle dataset)
# and split them per condition, so that the cache files expected by C2-C9 exist.
import glob
E_all   = np.load(glob.glob('/kaggle/input/**/embeddings_all.npy', recursive=True)[0])
ids_all = pd.read_parquet(glob.glob('/kaggle/input/**/embeddings_all_ids.parquet', recursive=True)[0])
E_m     = np.load(glob.glob('/kaggle/input/**/embeddings_masked_v2_all.npy', recursive=True)[0])
ids_m   = pd.read_parquet(glob.glob('/kaggle/input/**/embeddings_masked_v2_all_ids.parquet', recursive=True)[0])
pos_all = {r: i for i, r in enumerate(ids_all[ID])}
pos_m   = {r: i for i, r in enumerate(ids_m[ID])}
for c in CONDITIONS:
    ids_c = reviews.loc[reviews["cond_l"] == c, [ID]].reset_index(drop=True)
    np.save(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy", E_all[[pos_all[r] for r in ids_c[ID]]])
    ids_c.to_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet", index=False)
    np.save(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}.npy", E_m[[pos_m[r] for r in ids_c[ID]]])
    ids_c.to_parquet(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}_ids.parquet", index=False)
print("memory reconstructed embeddings for each condition and saved them to the output folder")

memory reconstructed embeddings for each condition and saved them to the output folder


## C2 · Sentence-BERT embeddings

**What it does:** `BioLORD-2023` (a Sentence-BERT model trained on medical concepts) reads a review and returns 768 numbers that represent its *meaning*. Reviews that say the same thing in different words ("I couldn't sleep" / "insomnia") end up close to each other; with TF-IDF they would not.

**Decisions:**
- **Model:** chosen because it is biomedical (it understands clinical terms such as *insomnia*, *libido* and *withdrawal*). It is larger and slower than the general-purpose `all-MiniLM-L6-v2` (the alternative that can be set in `MODEL_NAME`). This notebook does not compare the two models, so the choice rests on domain fit, not on a measured advantage. Limit: `model.max_seq_length` (printed below); longer reviews are truncated.
- **`normalize_embeddings=True`:** every vector has length 1, so the dot product of two vectors equals their cosine similarity.
- **Input text:** `review_text_clean`, with complete sentences, because the model needs the grammar and not the normalised text.
- **Two files are saved:** a `.npy` file (the numbers) **and** `_ids.parquet` (which `review_id` each row belongs to). Without the IDs file, the `.npy` is useless.
- **Cache:** if the file already exists, it is skipped, so the cell can be re-run without repeating hours of computation. The file name contains the model name. In the final run the embeddings were not recomputed here: they were loaded from the Kaggle dataset in the cell above, so this cell only loads the model and skips the encoding.

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME)
print(MODEL_NAME, '| dim:', model.get_sentence_embedding_dimension(), '| max_seq_length:', model.max_seq_length)

for c in CONDITIONS:
    fname = f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy"
    if os.path.exists(fname):
        print(c, "already done, skipping"); continue
    sub = reviews[reviews["cond_l"] == c]
    t = time.time()
    emb = model.encode(sub[TEXT].tolist(), batch_size=BATCH_SIZE,
                       show_progress_bar=True, normalize_embeddings=True)
    np.save(fname, emb)
    sub[[ID]].to_parquet(fname.replace(".npy", "_ids.parquet"))
    print(c, emb.shape, f"{time.time()-t:.0f}s")

# The remaining reviews (other conditions): A searches similar stories in all 128,663 reviews, not only in my four conditions
fname = f"{OUTPUT_DIR}/{EMB_OTHER}.npy"
if EMBED_ALL_OTHER_REVIEWS and not os.path.exists(fname):
    sub = reviews[~reviews["cond_l"].isin(CONDITIONS)]
    t = time.time()
    emb = model.encode(sub[TEXT].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
    np.save(fname, emb)
    sub[[ID]].to_parquet(fname.replace(".npy", "_ids.parquet"))
    print("other conditions", emb.shape, f"{time.time()-t:.0f}s")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.17k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FremyCompany/BioLORD-2023 | dim: 768 | max_seq_length: 128
depression already done, skipping
birth control already done, skipping
pain already done, skipping
acne already done, skipping


## C3 · Sanity check of the embeddings

For each condition I check: (1) the number of rows in the `.npy` file equals the number of IDs and the number of reviews of that condition, (2) there are no NaN values, (3) every vector has norm 1.
If something fails here, everything that follows (clusters, labels, A's LSH) would be misaligned.

In [5]:
check = []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet")
    n   = int((reviews["cond_l"] == c).sum())
    check.append({"condition": c, "rows_npy": E.shape[0], "dim": E.shape[1], "n_ids": len(ids), "n_reviews": n,
                  "has_nan": bool(np.isnan(E).any()),
                  "norm_ok": bool(np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-3)),
                  "ids_unique": bool(ids[ID].is_unique)})
check = pd.DataFrame(check); print(check.to_string(index=False))
assert (check.rows_npy == check.n_ids).all() and (check.n_ids == check.n_reviews).all(), "misaligned"
assert not check.has_nan.any() and check.norm_ok.all() and check.ids_unique.all()
print("OK")

    condition  rows_npy  dim  n_ids  n_reviews  has_nan  norm_ok  ids_unique
   depression      7109  768   7109       7109    False     True        True
birth control     19520  768  19520      19520    False     True        True
         pain      5187  768   5187       5187    False     True        True
         acne      4549  768   4549       4549    False     True        True
OK


## C4 · Topic terms for each cluster: class-based TF-IDF, written by hand

**Context:** B grouped the reviews of each condition into k-means clusters (100 in total: Birth Control 23, Depression 26, Pain 26, Acne 25) and listed the top-10 TF-IDF terms of each one in `cluster_profiles.csv`. Here every cluster is described **automatically and reproducibly** by the words (and word pairs) that distinguish it from the other clusters of the same condition, with **class-based TF-IDF (c-TF-IDF)**, the idea behind BERTopic.

**How it works:** all the reviews of a cluster are joined into one big document, the terms are counted, and each term gets

`score(c, t) = tf(c, t) · log(1 + A / f(t))`

- `tf(c, t)`: the share of the words of cluster `c` that are the term `t`.
- `f(t)`: how many times `t` appears in all clusters of the condition together.
- `A`: the mean number of words per cluster (it only sets the scale).

A term scores high if it is frequent in this cluster and rare in the others ("pimples" scores high, "effects" low). We keep the 5, 10 and 30 best terms; the 30 are the candidates for C5. The overlap with B's top-10 terms is saved as `word_overlap_with_B`. The formula is written with numpy; only the counting uses `CountVectorizer`.

**Finding:** the terms include generic words (*doctor, went*), the condition name and drug names (*prozac*), and so do B's own terms (for example "ve", from "I've"). We do not remove them with a hand-written list, because it would be arbitrary; we **measure** the share of terms that are drug or condition names (`share_name_terms_top10`) instead: 14.3% on average, and 51 of the 100 clusters have one in their top 5. The mean overlap with B's terms is 0.677.

In [6]:
def c_tf_idf(counts):
    """Class-based TF-IDF (the idea of BERTopic), written by hand.
    counts[c, t] = how many times term t appears in ALL reviews of cluster c (a cluster = one big document).
    score[c, t] = tf[c, t] * log(1 + A / f[t])
      tf[c, t] = counts[c, t] / total words of cluster c     (share of the cluster's words that is t)
      f[t]     = counts of t in all clusters together
      A        = mean number of words per cluster
    A term scores high when it is frequent in this cluster and rare in the other clusters."""
    counts = np.asarray(counts, dtype=float)
    tf = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    A = counts.sum(axis=1).mean()
    f = np.maximum(counts.sum(axis=0), 1)
    return tf * np.log(1 + A / f)

In [7]:
import re
from sklearn.feature_extraction.text import CountVectorizer

TEXT_TOPIC = "review_text_normalised" if "review_text_normalised" in reviews.columns else TEXT

# --- condition and drug names, taken from the data: they are NOT excluded, only used to MEASURE how many terms are names ---
NAME_WORDS = set()
for col in ["condition_name", "drug_name"]:
    if col in reviews.columns:
        for v in reviews.loc[reviews["cond_l"].isin(CONDITIONS), col].dropna().astype(str).str.lower().unique():
            NAME_WORDS.update(re.findall(r"[a-z]{4,}", v))
NAME_WORDS.update(w for c_ in CONDITIONS for w in re.findall(r"[a-z]{3,}", c_))
prof = pd.read_csv(find_input("cluster_profiles.csv"))                 # B's TF-IDF terms (section B7)
prof["condition"] = prof["condition"].str.lower().str.strip()

topic_rows = []
for c in CONDITIONS:
    df = (reviews[reviews["cond_l"] == c][[ID, TEXT_TOPIC]].merge(clusters[[ID, CLUSTER_COL]], on=ID)
          .reset_index(drop=True))
    cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5, max_df=0.5)
    X = cv.fit_transform(df[TEXT_TOPIC].fillna(""))
    cl_ids = np.sort(df[CLUSTER_COL].unique()); pos = {k: i for i, k in enumerate(cl_ids)}
    M = sp.csr_matrix((np.ones(len(df)), (df[CLUSTER_COL].map(pos).values, np.arange(len(df)))),
                      shape=(len(cl_ids), len(df)))
    W = c_tf_idf((M @ X).toarray())                                    # (n_clusters, n_terms)
    terms = np.array(cv.get_feature_names_out())
    for i, cl in enumerate(cl_ids):
        order_terms = np.argsort(W[i])[::-1]
        top = terms[order_terms[:10]]
        top30 = terms[order_terms[:30]]
        b_row = prof[(prof["condition"] == c) & (prof["cluster"] == cl)]
        b_words = set(w for t in b_row["top_10_terms"].iloc[0].split(" | ") for w in t.split()) if len(b_row) else set()
        my_words = set(w for t in top for w in t.split())
        topic_rows.append({"condition": c, "cluster": cl, "n_reviews": int((df[CLUSTER_COL] == cl).sum()),
                           "ctfidf_top5": " | ".join(top[:5]), "ctfidf_top10": " | ".join(top), "ctfidf_top30": " | ".join(top30),
                           "b_top10": " | ".join(t for t in (b_row["top_10_terms"].iloc[0].split(" | ") if len(b_row) else [])),
                           "word_overlap_with_B": round(len(my_words & b_words) / max(len(my_words | b_words), 1), 3)})
topics = pd.DataFrame(topic_rows)
# measured finding: share of the top-10 terms that are drug/condition names (taken from the data, no hand-written list)
topics["share_name_terms_top10"] = topics["ctfidf_top10"].apply(
    lambda s_: round(float(np.mean([any(w in NAME_WORDS for w in t.split()) for t in s_.split(" | ")])), 2))
run_stats["share_top10_terms_that_are_drug_or_condition_names"] = round(float(topics["share_name_terms_top10"].mean()), 3)
run_stats["clusters_with_name_in_top5"] = int(topics["ctfidf_top5"].apply(lambda s_: any(w in NAME_WORDS for t in s_.split(" | ") for w in t.split())).sum())
print("Terms that are drug or condition names (mean share):", run_stats["share_top10_terms_that_are_drug_or_condition_names"],
      "| clusters with at least one in the top 5:", run_stats["clusters_with_name_in_top5"], "of", len(topics))
topics.to_csv(f"{OUTPUT_DIR}/cluster_topics_ctfidf.csv", index=False)
run_stats["ctfidf_word_overlap_with_B_mean"] = round(float(topics["word_overlap_with_B"].mean()), 3)
print("Mean word overlap with B's terms:", run_stats["ctfidf_word_overlap_with_B_mean"])
print(topics[["condition", "cluster", "n_reviews", "ctfidf_top5", "word_overlap_with_B"]].to_string(index=False))

Terms that are drug or condition names (mean share): 0.143 | clusters with at least one in the top 5: 51 of 100
Mean word overlap with B's terms: 0.677
    condition  cluster  n_reviews                                                              ctfidf_top5  word_overlap_with_B
   depression        0        230                   cymbalta | depression | pain | taking cymbalta | years                0.800
   depression        1        249                       pristiq | taking pristiq | feel | taking | effects                0.800
   depression        2        208                      effexor | xr | effexor xr | depression | withdrawal                0.778
   depression        3        272                   wellbutrin | xl | wellbutrin xl | effects | depression                0.889
   depression        4        184                          mood | swings | mood swings | depression | feel                0.636
   depression        5        227           thoughts | suicidal | suicidal thoug

## C5 · Keywords for each cluster with Sentence-BERT (KeyBERT-style)

From the 30 c-TF-IDF candidates of each cluster (C4), this step chooses the **5 that best represent its meaning and are specific to it**, using the embeddings (the idea of KeyBERT, Grootendorst 2020).

1. Every candidate is embedded with the same model (BioLORD), so terms and reviews live in the same space.
2. **Specificity:** the cosine similarity of the candidate to the cluster centroid **minus** its similarity to the centroid of the whole condition. A term that describes the whole condition (such as "depression") is equally close to both and scores about 0.
3. **Diversity (MMR):** the terms are chosen one at a time, with `score = (1 − diversity) · specificity − diversity · highest similarity to an already chosen term` and `diversity = 0.5`, so the five are not variants of one word ("weight", "gained weight").

The first keyword is the most specific one; the order is the order of selection, not a strict ranking. The keywords are saved in `cluster_labels_sbert_auto.csv`, and the first three form `label_final` in `cluster_labels_final.csv`.

**What the specificity step changed:** the condition name disappeared from the keywords and more specific terms appeared (for example "degenerative disc"), but more filler words also got through ("time", "years", "really").

**Limits:** the candidates can only be terms already in the c-TF-IDF of C4, so generic words and drug names can slip in; the keywords are a suggestion, not a final label, and the last positions often hold filler words. The number of candidates (30) and the diversity (0.5) are design choices, not tuned. The metrics of C7 evaluate the c-TF-IDF terms and the clusters, not these keywords.

In [8]:
# Keyword selection: c-TF-IDF candidates re-ranked with BioLORD, specific to the cluster (centroid minus condition centroid) and diverse (MMR).
def keybert_rank(centroid, cand_vecs, cands, top_n=5, diversity=0.5, baseline=None):
    """Pick top_n candidate terms that are close to the cluster centroid, different from each other (MMR),
    and specific to the cluster: if `baseline` (centroid of the whole condition) is given, its similarity is subtracted.
    score(i) = (1 - diversity) * [cos(cand_i, centroid) - cos(cand_i, baseline)] - diversity * max cos(cand_i, already chosen)"""
    sims = cand_vecs @ centroid
    if baseline is not None:
        sims = sims - cand_vecs @ baseline
    chosen = [int(np.argmax(sims))]
    while len(chosen) < min(top_n, len(cands)):
        rest = [i for i in range(len(cands)) if i not in chosen]
        redundancy = (cand_vecs[rest] @ cand_vecs[chosen].T).max(axis=1)
        score = (1 - diversity) * sims[rest] - diversity * redundancy
        chosen.append(rest[int(np.argmax(score))])
    return [cands[i] for i in chosen]


# embeddings of all candidate terms (computed once)
all_cands = sorted({t for s_ in topics["ctfidf_top30"] for t in s_.split(" | ")})
cand_emb = dict(zip(all_cands, model.encode(all_cands, batch_size=BATCH_SIZE, normalize_embeddings=True)))

rows = []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True)
    ids["row"] = range(len(ids))
    df = (ids.merge(clusters[[ID, CLUSTER_COL]], on=ID)
             .sort_values("row").reset_index(drop=True))
    cond_centroid = E.mean(axis=0); cond_centroid /= np.linalg.norm(cond_centroid)      # centre of the whole condition
    for cl, idx in df.groupby(CLUSTER_COL).indices.items():
        centroid = E[df["row"].values[idx]].mean(axis=0); centroid /= np.linalg.norm(centroid)
        cands = topics[(topics["condition"] == c) & (topics["cluster"] == cl)]["ctfidf_top30"].iloc[0].split(" | ")
        top = keybert_rank(centroid, np.vstack([cand_emb[t] for t in cands]), cands,
                           top_n=5, diversity=0.5, baseline=cond_centroid)
        rows.append({"condition": c, "cluster": cl, "n_reviews": len(idx), "sbert_keywords": " | ".join(top)})
auto = pd.DataFrame(rows)
auto.to_csv(f"{OUTPUT_DIR}/cluster_labels_sbert_auto.csv", index=False)
print(auto.to_string(index=False))

    condition  cluster  n_reviews                                               sbert_keywords
   depression        0        230                 taking cymbalta | years | doctor | pain | im
   depression        1        249                        pristiq | better | ive | felt | years
   depression        2        208        effexor xr | years | withdrawal symptoms | ive | felt
   depression        3        272                      wellbutrin | years | xl | weight | good
   depression        4        184               mood swings | improved | felt | doctor | month
   depression        5        227                 suicidal thoughts | time | going | dont | im
   depression        6        365                      asleep | time | medication | ive | fall
   depression        7         88                  high blood | day | heart | weight | anxiety
   depression        8        163                    viibryd | diarrhea | sexual | far | tried
   depression        9        151                s

## C6 · Table of B's clusters

A summary of each of B's 100 k-means clusters. This is a helper step: it does not use the embeddings and it is not an evaluation. It builds one row per cluster with:

- `condition` and `cluster`: which cluster it is (for example, Birth Control, cluster 3).
- `n_reviews`: how many reviews it contains.
- `mean_rating`: the average rating of those reviews.
- `top_drugs`: the three drugs that appear most often in the cluster.

For each condition, the cell loads the review IDs, joins the drug and rating (from A) and the cluster of each review (from B), and groups by cluster.

**What it is used for:**
1. It is the base of the final table for D (C10.2, `cluster_labels_final.csv`), to which the keywords, the NPMI and the cohesion are added later, so that D receives everything in one file.
2. It shows at a glance whether a cluster is dominated by one drug (for example, `top_drugs` with a single name repeated), which supports the finding that some clusters group by drug rather than by experience.

In [9]:
rows = []
for c in CONDITIONS:
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet")
    df = (ids.merge(reviews[[ID, "drug_name", "rating"]], on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID))
    for cl, sub in df.groupby(CLUSTER_COL):
        rows.append({"condition": c, "cluster": cl, "n_reviews": len(sub), "mean_rating": round(sub["rating"].mean(), 2),
                     "top_drugs": ", ".join(sub["drug_name"].value_counts().head(3).index)})
to_label = pd.DataFrame(rows)
print(to_label.shape)   # (100, 5)
print(to_label.to_string(index=False))

(100, 5)
    condition  cluster  n_reviews  mean_rating                                                                                               top_drugs
   depression        0        230         7.27                                                                       Duloxetine, Vortioxetine, Abilify
   depression        1        249         7.51                                                                    Desvenlafaxine, Viibryd, Venlafaxine
   depression        2        208         7.27                                                                           Venlafaxine, Effexor, Viibryd
   depression        3        272         7.51                                                                    Wellbutrin, Wellbutrin XL, Bupropion
   depression        4        184         6.99                                                                  Citalopram, Desvenlafaxine, Sertraline
   depression        5        227         6.02                                       

## C7 · Automatic metrics for each cluster

We have 100 clusters and keywords for each, and we check **whether the clusters are good** with two numbers (no manual labelling).

**1. NPMI coherence: do the words of the cluster go together?** For every pair of the 10 most distinctive terms (C4) we check whether they appear **in the same reviews** more often than chance. NPMI goes from −1 to 1 (near 0 = chance, negative = rarely together), and the value of a cluster is the mean over its pairs. To know what counts as "high", we compute a **random baseline**: 200 random sets of 10 terms from the same condition.

**2. Cohesion gap, in meaning space: is the cluster separated?** For every review: the cosine similarity to the centre of its own cluster minus the similarity to the centre of the nearest other cluster of the same condition, averaged over the cluster. A clearly positive value means the cluster is well separated; about 0 means it overlaps its neighbours.

**Results of this run:**
- **NPMI** 0.137 against −0.165 for random sets (acne 0.139 / −0.131, birth control 0.146 / −0.079, depression 0.113 / −0.079, pain 0.151 / −0.360): the terms go together far more than chance in every condition.
- **Cohesion gap** −0.006, essentially zero: B's word-based clusters are **not separated** in the BioLORD space, which fits what C9 shows.

**Limitations:** the terms were chosen for being frequent in the cluster, which favours a high NPMI (the random baseline compensates only partly), and neither metric says whether a keyword makes sense to a person.

In [10]:
def npmi_of(X, eps=1e-12):
    # X: sparse binary matrix (reviews x terms). Mean NPMI over all pairs of terms.
    X = X.tocsc(); n = X.shape[0]; dfc = np.asarray(X.sum(axis=0)).ravel(); vals = []
    for i in range(X.shape[1]):
        for j in range(i + 1, X.shape[1]):
            pa, pb = dfc[i] / n, dfc[j] / n
            pab = float(X[:, i].multiply(X[:, j]).sum()) / n
            if pab <= 0 or pa == 0 or pb == 0: vals.append(-1.0)
            elif pab >= 1: vals.append(1.0)
            else: vals.append(np.log(pab / (pa * pb)) / (-np.log(pab)))
    return float(np.mean(vals)) if vals else float("nan")

rng = np.random.default_rng(0); met_rows = []
for c in CONDITIONS:
    E_  = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids_ = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True); ids_["row"] = range(len(ids_))
    d_ = (ids_.merge(reviews[[ID, TEXT]], on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID).sort_values("row").reset_index(drop=True))
    tc = topics[topics.condition == c]
    pool = sorted({t for s_ in tc["ctfidf_top30"] for t in s_.split(" | ")})
    cv_ = CountVectorizer(vocabulary=pool, stop_words="english", ngram_range=(1, 2), binary=True)
    Xall = cv_.fit_transform(d_[TEXT].astype(str)).tocsc(); col = {t: i for i, t in enumerate(pool)}
    rand_scores = []
    for _ in range(200):
        sel = list(rng.choice(pool, 10, replace=False))
        rand_scores.append(npmi_of(Xall[:, [col[t] for t in sel]]))
    cl_ids = sorted(d_[CLUSTER_COL].unique())
    cents = np.vstack([E_[d_.loc[d_[CLUSTER_COL] == cl, "row"].values].mean(axis=0) for cl in cl_ids])
    cents /= np.linalg.norm(cents, axis=1, keepdims=True)
    for k_, cl in enumerate(cl_ids):
        terms = [t for t in tc[tc.cluster == cl]["ctfidf_top10"].iloc[0].split(" | ") if t in col]
        coh = npmi_of(Xall[:, [col[t] for t in terms]])
        mine = E_[d_.loc[d_[CLUSTER_COL] == cl, "row"].values]
        own = float((mine @ cents[k_]).mean())
        others = np.delete(np.arange(len(cl_ids)), k_)
        nearest = float((mine @ cents[others].T).max(axis=1).mean()) if len(others) else float("nan")
        met_rows.append({"condition": c, "cluster": cl, "npmi": round(coh, 3), "npmi_random_mean": round(float(np.mean(rand_scores)), 3),
                         "cohesion_own": round(own, 3), "cohesion_gap": round(own - nearest, 3)})
metrics = pd.DataFrame(met_rows)
metrics.to_csv(f"{OUTPUT_DIR}/cluster_coherence_metrics.csv", index=False)
print(metrics.groupby("condition")[["npmi", "npmi_random_mean", "cohesion_gap"]].mean().round(3))
run_stats["npmi_mean_clusters"] = round(float(metrics["npmi"].mean()), 3)
run_stats["npmi_mean_random_baseline"] = round(float(metrics["npmi_random_mean"].mean()), 3)
run_stats["cohesion_gap_mean"] = round(float(metrics["cohesion_gap"].mean()), 3)
print("\nMean NPMI of the clusters:", run_stats["npmi_mean_clusters"], "| random baseline:", run_stats["npmi_mean_random_baseline"])

                npmi  npmi_random_mean  cohesion_gap
condition                                           
acne           0.139            -0.131        -0.018
birth control  0.146            -0.079         0.009
depression     0.113            -0.079        -0.007
pain           0.151            -0.360        -0.007

Mean NPMI of the clusters: 0.137 | random baseline: -0.165


## C8 · Why k-means and not DBSCAN

B ran two clustering algorithms on the same reviews: **k-means** assigns every review to one of k groups (23–26 per condition, 100 clusters in total), while **DBSCAN** looks for dense regions and labels everything else as **noise** (`-1`). This section explains why I describe only the k-means clusters; the cell below prints the share of noise of DBSCAN and the size of its clusters for each condition.

B's DBSCAN results do not suit this purpose:
- In every condition DBSCAN returns **one giant cluster plus a few micro-clusters** of 2 to 7 reviews (for example Birth Control: 17,282 reviews in cluster 0 and clusters of 7, 4 and 3). There are no separate experience types to describe.
- It also leaves **11 % to 26 % of the reviews as noise** (Birth Control 11.4 %, Depression 19.3 %, Pain 19.8 %, Acne 26.1 %), which means leaving those patients' experiences out of the analysis.
- k-means gives **full coverage**: every review belongs to a cluster that can be described.

Therefore I work with the 100 k-means clusters.


In [11]:
print("DBSCAN noise share (-1) per condition:")
print(clusters.groupby(COND)["dbscan_cluster"].apply(lambda s: (s == -1).mean().round(3)))
print("\nNumber of distinct values of dbscan_cluster:")
print(clusters.groupby(COND)["dbscan_cluster"].nunique())
for c, g in clusters.groupby(COND):
    print(c, g["dbscan_cluster"].value_counts().head(5).to_dict())

DBSCAN noise share (-1) per condition:
condition_name
Acne             0.261
Birth Control    0.114
Depression       0.193
Pain             0.198
Name: dbscan_cluster, dtype: float64

Number of distinct values of dbscan_cluster:
condition_name
Acne             4
Birth Control    9
Depression       3
Pain             3
Name: dbscan_cluster, dtype: int64
Acne {0: 3354, -1: 1187, 1: 4, 2: 4}
Birth Control {0: 17282, -1: 2217, 5: 7, 3: 4, 2: 3}
Depression {0: 5732, -1: 1375, 1: 2}
Pain {0: 4152, -1: 1029, 1: 6}


## C9 · Clustering in meaning space: does Sentence-BERT group by experience or by drug?

**Question:** if I re-cluster the reviews with Sentence-BERT embeddings (using the same k as B for each condition), do the clusters differ from B's word-based clusters, and do they describe experiences or just drugs?

**Metric: drug purity.** For each cluster, the share of its reviews that belong to its most frequent `drug_name`, averaged over the clusters. A high value means the cluster is basically "one drug". The baseline is the share of the most frequent drug in the whole condition: if the purity only matches this baseline, the clustering does not depend on the drug.

**Other metrics:**
- **ARI** (adjusted Rand index): how much my clusters agree with B's (1 = identical, 0 = as expected by chance).
- **Davies-Bouldin index:** how well separated the clusters are (lower is better). It is computed for B's clusters *inside the embedding space* (`DB_B_in_sbert_space`) and for the Sentence-BERT k-means clusters (`DB_sbert`), so both are compared with the same yardstick. Note that B computed its own DB in the TF-IDF/SVD space, where the numbers are not comparable with the ones here.

**Two versions:**
1. **Unmasked Sentence-BERT.** Compare `purity_sbert_unmasked` with `baseline_top_drug_share` to see how much the grouping depends on the drug.
2. **Sentence-BERT with drug names replaced by `medication`.** The list of names is built **only from project data** (`drug_name`, `drug_name_aliases`, `brand_generic_drug_name_pairs.csv`, `drug_families.csv`); nothing is written from memory.

**Limits:** names that are not in those files (for example some brands) are not masked, and the "residual" share printed at the end of the masked cell is computed with the same list, so it cannot detect names missing from it: it is not evidence that masking is complete. One ordinary word that appears in the name files (`commit`, as in "commit suicide") is removed from the list so that it is not masked. There is also no "ground truth" to say which clustering is better, and the numbers depend on the masking list. I present this as a **finding and a limitation**, not as a demonstrated improvement.

In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score

def drug_purity(df, col):
    return df.groupby(col)["drug_name"].agg(lambda s: s.value_counts(normalize=True).iloc[0]).mean()

# --- version 1: SBERT unmasked ---
res_unmasked, assign_unmasked = [], []
for c in CONDITIONS:
    E   = np.load(f"{OUTPUT_DIR}/{EMB}_{key(c)}.npy")
    ids = pd.read_parquet(f"{OUTPUT_DIR}/{EMB}_{key(c)}_ids.parquet").reset_index(drop=True); ids["row"] = range(len(ids))
    df = (reviews[reviews["cond_l"] == c].merge(ids, on=ID).merge(clusters[[ID, CLUSTER_COL]], on=ID)
          .sort_values("row").reset_index(drop=True))
    E = E[df["row"].values]
    k = df[CLUSTER_COL].nunique()
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(E)
    df["sbert_cluster"] = km.labels_
    res_unmasked.append({"condition": c, "k": k,
        "baseline_top_drug_share": round(df["drug_name"].value_counts(normalize=True).iloc[0], 3),
        "purity_B": round(drug_purity(df, CLUSTER_COL), 3),
        "purity_sbert_unmasked": round(drug_purity(df, "sbert_cluster"), 3),
        "DB_sbert": round(davies_bouldin_score(E, km.labels_), 3),
        "DB_B_in_sbert_space": round(davies_bouldin_score(E, df[CLUSTER_COL]), 3),
        "ARI_vs_B": round(adjusted_rand_score(df[CLUSTER_COL], df["sbert_cluster"]), 3),
        "rating_spread_B": round(df.groupby(CLUSTER_COL)["rating"].mean().std(), 3),
        "rating_spread_sbert": round(df.groupby("sbert_cluster")["rating"].mean().std(), 3)})
    assign_unmasked.append(df[[ID, COND, "sbert_cluster"]])
res_unmasked = pd.DataFrame(res_unmasked)
res_unmasked.to_csv(f"{OUTPUT_DIR}/sbert_vs_tfidf_summary.csv", index=False)
pd.concat(assign_unmasked).to_parquet(f"{OUTPUT_DIR}/sbert_cluster_assignments.parquet")
print(res_unmasked.to_string(index=False))

    condition  k  baseline_top_drug_share  purity_B  purity_sbert_unmasked  DB_sbert  DB_B_in_sbert_space  ARI_vs_B  rating_spread_B  rating_spread_sbert
   depression 26                    0.093     0.267                  0.484     3.056                7.856     0.101            1.027                1.052
birth control 23                    0.148     0.423                  0.439     3.346                6.122     0.163            1.158                1.395
         pain 26                    0.085     0.260                  0.293     3.435                8.577     0.068            1.009                0.862
         acne 25                    0.121     0.321                  0.429     3.312                8.302     0.082            1.079                1.264


In [13]:
# --- list of drug names built ONLY from the project data ---
from collections import Counter
pairs = pd.read_csv(find_input("brand_generic_drug_name_pairs.csv"))
fam   = pd.read_csv(find_input("drug_families.csv"))

def split_names(v):
    for part in re.split(r"\s*\|\s*|\s*/\s*", str(v)):      # the files separate names with " | " and " / "
        p = part.strip().lower()
        if len(p) >= 4:
            yield p

names = set()
for _, r in reviews.drop_duplicates("drug_name")[["drug_name", "drug_name_aliases"]].iterrows():
    for n in [r["drug_name"]] + list(r["drug_name_aliases"]):
        names.update(split_names(n))
for t in (pairs, fam):
    for v in t["drug_names"].dropna():
        names.update(split_names(v))

EXCLUDE = {"commit"}     # ordinary word ("commit suicide") that appears in the name files and must not be masked
names -= EXCLUDE
pattern = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(names, key=len, reverse=True)) + r")\b",
                     flags=re.IGNORECASE)
mask = lambda t: pattern.sub("medication", t)

cuatro = reviews[reviews["cond_l"].isin(CONDITIONS)]
cnt = Counter(m.lower() for t in cuatro[TEXT] for m in pattern.findall(t))
print(len(names), "names in the list;", len(cnt), "appear in the 4 conditions")
print("Most masked names (check that none is an ordinary word):")
print(cnt.most_common(60))

3460 names in the list; 708 appear in the 4 conditions
Most masked names (check that none is an ordinary word):
[('nexplanon', 2195), ('mirena', 2074), ('implanon', 1720), ('skyla', 1321), ('nuvaring', 1081), ('zoloft', 958), ('lexapro', 915), ('accutane', 914), ('wellbutrin', 787), ('prozac', 769), ('pristiq', 732), ('cymbalta', 726), ('sprintec', 723), ('epiduo', 705), ('ibuprofen', 681), ('effexor', 634), ('tramadol', 625), ('celexa', 578), ('paragard', 566), ('percocet', 461), ('paxil', 400), ('yasmin', 385), ('lutera', 381), ('vicodin', 377), ('doxycycline', 366), ('viibryd', 366), ('norco', 364), ('aviane', 352), ('oxycodone', 352), ('oxycontin', 337), ('trinessa', 334), ('apri', 303), ('morphine', 297), ('tylenol', 293), ('dilaudid', 284), ('citalopram', 259), ('methadone', 258), ('nucynta', 250), ('xanax', 249), ('aczone', 243), ('minocycline', 242), ('copper', 238), ('abilify', 238), ('sertraline', 236), ('tri-sprintec', 234), ('advil', 231), ('beyaz', 224), ('hydrocodone', 22

In [14]:
# --- version 2: SBERT with drug names masked (slow the first time with BioLORD; afterwards the .npy is reused) ---
res_masked, assign_masked = [], []
for c in CONDITIONS:
    df = (reviews[reviews["cond_l"] == c].merge(clusters[[ID, CLUSTER_COL]], on=ID).reset_index(drop=True))
    f = f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}.npy"
    if os.path.exists(f):
        E = np.load(f)
    else:
        E = model.encode(df[TEXT].map(mask).tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        np.save(f, E)
    df[[ID]].to_parquet(f"{OUTPUT_DIR}/{EMB_MASKED}_{key(c)}_ids.parquet")
    k = df[CLUSTER_COL].nunique()
    df["sbert_masked_cluster"] = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(E)
    res_masked.append({"condition": c, "k": k,
        "baseline_top_drug_share": round(df["drug_name"].value_counts(normalize=True).iloc[0], 3),
        "purity_B": round(drug_purity(df, CLUSTER_COL), 3),
        "purity_sbert_masked_v2": round(drug_purity(df, "sbert_masked_cluster"), 3),
        "ARI_vs_B": round(adjusted_rand_score(df[CLUSTER_COL], df["sbert_masked_cluster"]), 3),
        "rating_spread_B": round(df.groupby(CLUSTER_COL)["rating"].mean().std(), 3),
        "rating_spread_masked": round(df.groupby("sbert_masked_cluster")["rating"].mean().std(), 3)})
    assign_masked.append(df[[ID, COND, "sbert_masked_cluster"]])
res_masked = pd.DataFrame(res_masked)
res_masked.to_csv(f"{OUTPUT_DIR}/sbert_masked_v2_summary.csv", index=False)
pd.concat(assign_masked).to_parquet(f"{OUTPUT_DIR}/sbert_masked_v2_cluster_assignments.parquet")
print(res_masked.to_string(index=False))

# residual share of reviews that still contain a name from the same list after masking (circular check: see the limits in C9)
leak = cuatro[TEXT].map(mask).str.lower().map(lambda t: bool(pattern.search(t))).mean()
print(f"\nReviews with a drug name still visible after masking: {leak:.1%}")

    condition  k  baseline_top_drug_share  purity_B  purity_sbert_masked_v2  ARI_vs_B  rating_spread_B  rating_spread_masked
   depression 26                    0.093     0.267                   0.143     0.026            1.027                 1.635
birth control 23                    0.148     0.423                   0.255     0.087            1.158                 1.691
         pain 26                    0.085     0.260                   0.151     0.041            1.009                 0.866
         acne 25                    0.121     0.321                   0.311     0.036            1.079                 1.352

Reviews with a drug name still visible after masking: 0.0%


**How to read C9:** compare `purity_sbert_unmasked` with `baseline_top_drug_share` (the baseline): if it is well above the baseline, the model groups by drug. Then compare `purity_sbert_masked_v2`: if it drops, masking the names reduces the dependence on the drug name. A low ARI against B means that the two clusterings are different. This does **not** show that one is better; it only shows that Sentence-BERT and TF-IDF find different structures.

**Results of this run (100 clusters; k = 26, 23, 26, 25 for Depression, Birth Control, Pain, Acne):**
- **Drug purity, unmasked:** Sentence-BERT 0.484 / 0.439 / 0.293 / 0.429 against B's 0.267 / 0.423 / 0.260 / 0.321 and baselines of 0.093 / 0.148 / 0.085 / 0.121 (same order). Both methods are above the baseline, and Sentence-BERT is higher than B in all four conditions, so **the embeddings group by drug even more than B's TF-IDF clusters**.
- **Drug purity, masked:** it drops to 0.143 / 0.255 / 0.151 / 0.311, much closer to the baselines but still above them. Masking removes most, not all, of the dependence on the drug.
- **ARI against B:** 0.07–0.16 unmasked and 0.03–0.09 masked, so the two clusterings are very different.
- **Davies-Bouldin:** 3.1–3.4 for the Sentence-BERT k-means against 6.1–8.6 for B's clusters measured in the same space. This comparison is circular, because k-means optimises exactly this space; it is not evidence that Sentence-BERT is better.
- **Rating spread** (standard deviation of the mean rating between clusters): unmasked, Sentence-BERT is similar to or higher than B (1.052 / 1.395 / 0.862 / 1.264 against 1.027 / 1.158 / 1.009 / 1.079); masked it is 1.635 / 1.691 / 0.866 / 1.352. In most conditions the Sentence-BERT clusters therefore differ at least as much in mean rating as B's, but this does not show that they separate good from bad experiences.

B's clusters also partly group by drug (`purity_B` > baseline), so this is not a weakness of Sentence-BERT alone.

## C10 · Deliverables

### C10.1 Embeddings for A (similarity search by meaning)

One file with **one vector per `review_id`**, in two versions: unmasked (`embeddings_all`) and with drug names masked (`embeddings_masked_v2_all`, four conditions only). The file written by this run covers only the **36,365 reviews of the four conditions**; the full 128,663-review file is in the Kaggle dataset linked in the README. Unmasked vectors contain the drug name, so a similarity search on them tends to pair reviews of the same drug.

In [15]:
def concat_embeddings(prefix, out_name, extra=()):
    E_all, id_all = [], []
    for c in CONDITIONS:
        E_all.append(np.load(f"{OUTPUT_DIR}/{prefix}_{key(c)}.npy"))
        id_all.append(pd.read_parquet(f"{OUTPUT_DIR}/{prefix}_{key(c)}_ids.parquet")[[ID]])
    for name in extra:                                   # reviews of other conditions (if they exist)
        if os.path.exists(f"{OUTPUT_DIR}/{name}.npy"):
            E_all.append(np.load(f"{OUTPUT_DIR}/{name}.npy"))
            id_all.append(pd.read_parquet(f"{OUTPUT_DIR}/{name}_ids.parquet")[[ID]])
    E_all = np.vstack(E_all).astype("float32"); ids_all = pd.concat(id_all, ignore_index=True)
    assert len(E_all) == len(ids_all) and ids_all[ID].is_unique
    np.save(f"{OUTPUT_DIR}/{out_name}.npy", E_all)
    ids_all.to_parquet(f"{OUTPUT_DIR}/{out_name}_ids.parquet")
    return E_all.shape

shape_unmasked = concat_embeddings(EMB, "embeddings_all", extra=[EMB_OTHER])
shape_masked   = concat_embeddings(EMB_MASKED, "embeddings_masked_v2_all")   # only the 4 conditions
print("unmasked:", shape_unmasked, "| masked v2:", shape_masked)
if shape_unmasked[0] < len(reviews):
    print(f"WARNING: embeddings_all covers {shape_unmasked[0]:,} of {len(reviews):,} reviews: tell A.")

unmasked: (36365, 768) | masked v2: (36365, 768)


### C10.2 Final labels for D

A CSV with **one row per cluster of B** (`cluster_labels_final.csv`), with these columns:
- `condition` (lower case), `cluster`, `n_reviews`, `mean_rating` and `top_drugs` (from C6).
- `sbert_keywords`: the five Sentence-BERT keywords of the cluster (C5).
- `ctfidf_top5`: the five best c-TF-IDF terms (C4).
- `npmi` and `cohesion_gap`: the automatic quality metrics of the cluster (C7).
- `label_final`: the first three Sentence-BERT keywords, joined with " / ". It is a keyword label chosen automatically, not a sentence written by a person (`label_source` says so).
- `b_top_terms`: B's top TF-IDF terms for the same cluster, so that both views can be compared.

In [16]:
base = to_label[["condition", "cluster", "n_reviews", "mean_rating", "top_drugs"]].merge(
    auto[["condition", "cluster", "sbert_keywords"]], on=["condition", "cluster"]).merge(
    topics[["condition", "cluster", "ctfidf_top5"]], on=["condition", "cluster"], how="left").merge(
    metrics[["condition", "cluster", "npmi", "cohesion_gap"]], on=["condition", "cluster"], how="left")
base["label_final"] = base["sbert_keywords"].str.split(" | ", regex=False).str[:3].str.join(" / ")
base["label_source"] = "auto_keywords_sbert"
prof = pd.read_csv(find_input("cluster_profiles.csv"))          # B's TF-IDF terms (section B7)
prof["condition"] = prof["condition"].str.lower().str.strip()
base = base.merge(prof[["condition", "cluster", "top_5_terms"]].rename(columns={"top_5_terms": "b_top_terms"}),
                  on=["condition", "cluster"], how="left")
base.to_csv(f"{OUTPUT_DIR}/cluster_labels_final.csv", index=False)
print(base[["condition", "cluster", "n_reviews", "label_final", "npmi", "cohesion_gap"]].to_string(index=False))

    condition  cluster  n_reviews                              label_final   npmi  cohesion_gap
   depression        0        230         taking cymbalta / years / doctor  0.081         0.112
   depression        1        249                   pristiq / better / ive  0.071         0.138
   depression        2        208 effexor xr / years / withdrawal symptoms  0.156         0.090
   depression        3        272                  wellbutrin / years / xl  0.124         0.084
   depression        4        184            mood swings / improved / felt  0.092        -0.049
   depression        5        227         suicidal thoughts / time / going  0.097        -0.037
   depression        6        365               asleep / time / medication  0.113        -0.031
   depression        7         88                 high blood / day / heart  0.205        -0.035
   depression        8        163              viibryd / diarrhea / sexual  0.123         0.084
   depression        9        151       

### C10.3 Run statistics

I save every number that I will quote in the report into one JSON file, `role_c_run_stats.json`, in the same way as A and B. It includes the model and the embedding size, the number of reviews embedded, the results of C4 (overlap with B's terms and the share of keywords that are drug or condition names), C7 (NPMI of the clusters against the random baseline, and the cohesion gap), C9 (drug purity, ARI and Davies-Bouldin, with and without drug names), the share of drug names left unmasked, the DBSCAN noise share, the DBSCAN noise share and the run time. (`model_comparison_sample` is empty because no model comparison is run in this notebook.)

In [17]:
run_stats.update({
  "model": MODEL_NAME, "embedding_dim": int(shape_unmasked[1]),
  "n_reviews_embedded": int(shape_unmasked[0]), "conditions": CONDITIONS,
  "n_clusters": int(len(to_label)),
  "sbert_unmasked": res_unmasked.to_dict(orient="records"),
  "sbert_masked_v2": res_masked.to_dict(orient="records"),
  "masked_names_in_list": int(len(names)),
  "residual_unmasked_name_share": round(float(leak), 3),
  "dbscan_noise_share": clusters.groupby(COND)["dbscan_cluster"].apply(lambda s: round(float((s == -1).mean()), 3)).to_dict(),
  "model_comparison_sample": model_comparison.to_dict(orient="records") if "model_comparison" in globals() else None,
  "runtime_minutes": round((time.perf_counter() - NOTEBOOK_STARTED) / 60, 1)})
stats = run_stats
(OUTPUT_DIR / "role_c_run_stats.json").write_text(json.dumps(stats, indent=2, default=str))
print(json.dumps(stats, indent=1, default=str)[:3000])

{
 "n_reviews_all": 128663,
 "n_reviews_in_scope": 36365,
 "share_top10_terms_that_are_drug_or_condition_names": 0.143,
 "clusters_with_name_in_top5": 51,
 "ctfidf_word_overlap_with_B_mean": 0.677,
 "npmi_mean_clusters": 0.137,
 "npmi_mean_random_baseline": -0.165,
 "cohesion_gap_mean": -0.006,
 "model": "FremyCompany/BioLORD-2023",
 "embedding_dim": 768,
 "n_reviews_embedded": 36365,
 "conditions": [
  "depression",
  "birth control",
  "pain",
  "acne"
 ],
 "n_clusters": 100,
 "sbert_unmasked": [
  {
   "condition": "depression",
   "k": 26,
   "baseline_top_drug_share": 0.093,
   "purity_B": 0.267,
   "purity_sbert_unmasked": 0.484,
   "DB_sbert": 3.056,
   "DB_B_in_sbert_space": 7.856,
   "ARI_vs_B": 0.101,
   "rating_spread_B": 1.027,
   "rating_spread_sbert": 1.052
  },
  {
   "condition": "birth control",
   "k": 23,
   "baseline_top_drug_share": 0.148,
   "purity_B": 0.423,
   "purity_sbert_unmasked": 0.439,
   "DB_sbert": 3.346,
   "DB_B_in_sbert_space": 6.122,
   "ARI_vs_B": 

## Outputs, usage and limitations

The list of output files, how to load them in another notebook, how to run this notebook and its limitations are in the **README**. All the numbers of this run are in `outputs/role_c_run_stats.json`.